# Equation Detection and LaTeX Recognition Pipeline
**Author:** Juan Esteban Agudelo Ortiz  
**Email:** juan.es.agor@gmail.com

---

This notebook implements an equation detection and recognition pipeline
that locates mathematical expressions in PDF pages and handwritten note
images, then converts them to LaTeX source code. The output extends the
`IngestedDocument` structure defined in the ingestion pipeline notebook
with a new `equations` field.

For PDFs, detection uses a cascade of three strategies: font-based
detection, numbering pattern detection, and vertical spacing as fallback.
For handwritten note images, detection is purely visual using pix2tex
directly on image regions.

Recognition uses pix2tex (LaTeX-OCR), a vision model that takes an image
crop of an equation and produces the corresponding LaTeX code.

### Limitations
1. Font-based detection only works on text-native PDFs; scanned PDFs require OCR preprocessing.
2. pix2tex accuracy degrades significantly on handwritten equations and non-standard notation.
3. Complex multi-line equations are sometimes split into multiple detections.
4. Detection recall is not guaranteed; some equations may be missed by all three strategies.
5. Handwritten equation recognition is best-effort; results should be reviewed manually before use.

## 0. Install dependencies

In [18]:
# Run only once
# !pip install pix2tex pillow pymupdf numpy

## 1. Imports and configuration

In [19]:
import re
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import fitz
import numpy as np
from PIL import Image
from pix2tex.cli import LatexOCR

# --- Directory setup ---
BASE_DIR    = Path("..")
UPLOADS_DIR = BASE_DIR / "data" / "uploads"
OUT_DIR     = BASE_DIR / "outputs"

UPLOADS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Reproduce IngestedDocument from notebook 1 ---
from enum import Enum

class InputType(Enum):
    PDF           = "pdf"
    HANDWRITTEN   = "handwritten_image"
    REFERENCE_IMG = "reference_image"
    PLAIN_TEXT    = "plain_text"

@dataclass
class IngestedDocument:
    input_type       : InputType
    text             : str
    reference_images : list = field(default_factory=list)
    source_path      : Optional[Path] = None

print(f"Uploads dir : {UPLOADS_DIR.resolve()}")
print(f"Outputs dir : {OUT_DIR.resolve()}")
print("Configuration ready ✓")

Uploads dir : /home/juanessao/Documents/Datos_de_Ciencia/playlists/hugging-face-hackaton/flashcard-generator-slm/data/uploads
Outputs dir : /home/juanessao/Documents/Datos_de_Ciencia/playlists/hugging-face-hackaton/flashcard-generator-slm/outputs
Configuration ready ✓


## 2. Equation detection in PDF pages

Detection uses a cascade of three strategies applied in order:

1. **Font-based:** blocks containing mathematical fonts (e.g. Computer Modern Math)
   are flagged as equations. Most reliable for text-native PDFs.
2. **Numbering pattern:** blocks adjacent to patterns like `(21.1)` or `Eq. 3`
   are flagged as equations. Depends on author conventions.
3. **Spacing-based:** blocks with vertical margins significantly larger than the
   median block spacing are flagged as equations. Used as fallback.

A block is included if it is flagged by at least one strategy. The crop
around each detected block is extracted as a PIL Image and passed to the
LaTeX-OCR model in Section 3.

In [20]:
def detect_equation_blocks(page: fitz.Page) -> list[dict]:
    """
    Detect blocks likely containing equations using a three-strategy cascade.

    Parameters
    ----------
    page : fitz.Page
        A single page from a PyMuPDF document.

    Returns
    -------
    list[dict]
        List of detected equation blocks, each with keys:
        - 'bbox': bounding box (x0, y0, x1, y1)
        - 'strategy': which strategy flagged it
    """
    blocks = page.get_text("dict")["blocks"]
    detected = []
    seen_bboxes = set()

    # Compute median vertical spacing between blocks for strategy 3
    y_positions = [b["bbox"][1] for b in blocks if "lines" in b]
    spacings = [y_positions[i+1] - y_positions[i] 
                for i in range(len(y_positions) - 1)]
    median_spacing = float(np.median(spacings)) if spacings else 0.0

    for block in blocks:
        if "lines" not in block:
            continue

        bbox = tuple(block["bbox"])
        flagged = False
        strategy = None

        # Strategy 1: font-based detection
        for line in block["lines"]:
            for span in line["spans"]:
                font = span["font"].lower()
                if any(math_font in font for math_font in 
                       ["math", "symbol", "cmmi", "cmsy", "msam", "msbm"]):
                    flagged = True
                    strategy = "font"
                    break
            if flagged:
                break

        # Strategy 2: numbering pattern detection
        if not flagged:
            block_text = " ".join(
                span["text"] for line in block["lines"] 
                for span in line["spans"]
            )
            if re.search(r"\(\d+\.\d+\)|Eq\.\s*\d+|Equation\s*\d+", block_text):
                flagged = True
                strategy = "numbering"

        # Strategy 3: spacing-based detection
        if not flagged and median_spacing > 0:
            block_y0 = block["bbox"][1]
            block_y1 = block["bbox"][3]
            block_height = block_y1 - block_y0

            # Count words in block to filter out long text blocks
            block_text = " ".join(
                span["text"] for line in block["lines"]
                for span in line["spans"]
            )
            word_count = len(block_text.split())

            if block_height > 4.0 * median_spacing and word_count <= 30:
                flagged = True
                strategy = "spacing"

        if flagged and bbox not in seen_bboxes:
            detected.append({"bbox": bbox, "strategy": strategy})
            seen_bboxes.add(bbox)

    return detected


print("detect_equation_blocks defined ✓")

detect_equation_blocks defined ✓


## 3. LaTeX-OCR model

pix2tex (LaTeX-OCR) is a vision model that takes an image crop of an
equation and produces the corresponding LaTeX code. It uses a
ViT (Vision Transformer) encoder to extract visual features from the
image, and a transformer decoder to generate the LaTeX token sequence.

The model takes as input an image crop of dimensions $(H, W)$ and
produces a sequence of LaTeX tokens $\hat{y} = (y_1, y_2, ..., y_n)$
where each $y_i$ is a LaTeX token such as `\frac`, `\sum`, or a variable
name. The generation stops when the model produces an end-of-sequence
token.

Each detected equation block is cropped from the page image at 300 DPI
before being passed to the model. Higher DPI produces sharper crops and
improves recognition accuracy.

In [21]:
# Initialize model once — downloads weights on first run (~1GB)
latex_ocr = LatexOCR()


def crop_equation_from_pdf(page: fitz.Page, bbox: tuple, dpi: int = 300) -> Image.Image:
    """
    Crop an equation region from a PDF page as a PIL Image.

    Parameters
    ----------
    page : fitz.Page
        A single page from a PyMuPDF document.
    bbox : tuple
        Bounding box (x0, y0, x1, y1) of the equation block.
    dpi : int
        Resolution for page rendering. Default 300.

    Returns
    -------
    PIL.Image.Image
        Cropped equation image in RGB mode.
    """
    scale = dpi / 72
    mat = fitz.Matrix(scale, scale)
    clip = fitz.Rect(bbox)
    pix = page.get_pixmap(matrix=mat, clip=clip)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    return img


def recognize_equation(crop: Image.Image) -> str:
    """
    Convert an equation image crop to LaTeX source code.

    Parameters
    ----------
    crop : PIL.Image.Image
        Image of the equation region.

    Returns
    -------
    str
        LaTeX source code recognized from the image.
    """
    return latex_ocr(crop)


print("LaTeX-OCR model initialized ✓")

LaTeX-OCR model initialized ✓


## 4. Post-processing and validation of LaTeX output

The raw output of pix2tex may contain formatting artifacts that need
to be cleaned before use. Additionally, not every recognized string
is valid LaTeX; a validation step filters out low-quality recognitions.

A recognized string is accepted if it meets two conditions:
1. It contains at least one LaTeX math token (e.g. `\frac`, `\sum`, `^`, `_`)
2. Its length exceeds a minimum character threshold $n_{min}$

Strings that fail either condition are discarded and logged as
failed recognitions for manual review.

In [22]:
# Minimum requirements for a valid LaTeX recognition
MIN_LATEX_CHARS = 3
MATH_TOKENS = re.compile(r"\\[a-zA-Z]+|[\^_\{\}]|\d+/\d+")


def clean_latex(raw: str) -> str:
    """
    Clean raw pix2tex output by removing common formatting artifacts.

    Parameters
    ----------
    raw : str
        Raw LaTeX string from pix2tex.

    Returns
    -------
    str
        Cleaned LaTeX string.
    """
    # Remove surrounding whitespace
    latex = raw.strip()

    # Remove redundant math delimiters if present
    for delim in ["$$", "$"]:
        if latex.startswith(delim) and latex.endswith(delim):
            latex = latex[len(delim):-len(delim)].strip()

    return latex


def validate_latex(latex: str, min_chars: int = MIN_LATEX_CHARS) -> bool:
    """
    Validate that a LaTeX string represents a plausible mathematical expression.

    Parameters
    ----------
    latex : str
        Cleaned LaTeX string.
    min_chars : int
        Minimum character length required.

    Returns
    -------
    bool
        True if the string passes validation, False otherwise.
    """
    if len(latex.strip()) < min_chars:
        return False
    if not MATH_TOKENS.search(latex):
        return False
    return True


# Quick test
test_cases = [
    ("\\frac{1}{2}",   True),
    ("Figure 21.3",    False),
    ("x^2 + y^2",      True),
    ("",               False),
    ("$$\\sum_{i=0}^{n} x_i$$", True),
]

for raw, expected in test_cases:
    cleaned = clean_latex(raw)
    result  = validate_latex(cleaned)
    status  = "✓" if result == expected else "✗"
    print(f"{status} '{raw[:30]:30s}' → valid={result}")

✓ '\frac{1}{2}                   ' → valid=True
✓ 'Figure 21.3                   ' → valid=False
✓ 'x^2 + y^2                     ' → valid=True
✓ '                              ' → valid=False
✓ '$$\sum_{i=0}^{n} x_i$$        ' → valid=True


## 5. Integration with IngestedDocument

This section extends `IngestedDocument` with a new `equations` field
using inheritance. `IngestedDocumentWithEquations` adds the detected
and validated LaTeX expressions without modifying the original dataclass.

Each detected equation is stored as a dict with three fields:

| Field | Type | Description |
|---|---|---|
| `latex` | `str` | validated LaTeX source code |
| `bbox` | `tuple` | bounding box of the equation in the page |
| `strategy` | `str` | detection strategy that flagged it |
| `page` | `int` | page number where the equation was found |

In [23]:
@dataclass
class IngestedDocumentWithEquations(IngestedDocument):
    """
    Extends IngestedDocument with detected and validated LaTeX equations.

    Additional attributes
    --------------------
    equations : list[dict]
        Each entry contains:
        - 'latex'    : str   — validated LaTeX source code
        - 'bbox'     : tuple — bounding box (x0, y0, x1, y1) in the page
        - 'strategy' : str   — detection strategy that flagged the block
        - 'page'     : int   — zero-based page number
    """
    equations: list = field(default_factory=list)


def extract_equations_from_pdf(
    doc: IngestedDocument,
    max_pages: int = None
) -> IngestedDocumentWithEquations:
    """
    Detect and recognize equations in a PDF IngestedDocument.

    Parameters
    ----------
    doc : IngestedDocument
        Must have input_type == InputType.PDF.
    max_pages : int, optional
        Maximum number of pages to process. None means all pages.

    Returns
    -------
    IngestedDocumentWithEquations
        Extended document with detected equations.
    """
    if doc.input_type != InputType.PDF:
        raise ValueError(f"Expected PDF input, got {doc.input_type.value}")

    pdf = fitz.open(doc.source_path)
    equations = []
    pages_to_process = list(pdf.pages())[:max_pages]

    for page_num, page in enumerate(pages_to_process):
        detected = detect_equation_blocks(page)

        for eq_block in detected:
            crop = crop_equation_from_pdf(page, eq_block["bbox"])
            raw  = recognize_equation(crop)

            cleaned = clean_latex(raw)
            if not validate_latex(cleaned):
                continue

            equations.append({
                "latex"    : cleaned,
                "bbox"     : eq_block["bbox"],
                "strategy" : eq_block["strategy"],
                "page"     : page_num,
            })

    pdf.close()

    return IngestedDocumentWithEquations(
        input_type       = doc.input_type,
        text             = doc.text,
        reference_images = doc.reference_images,
        source_path      = doc.source_path,
        equations        = equations,
    )


print("extract_equations_from_pdf defined ✓")

extract_equations_from_pdf defined ✓


## 6. Demo — Equations from OpenStax chapter 21

This demo runs the full equation detection and recognition pipeline on
the carboxylic acid derivatives chapter from OpenStax Organic Chemistry.

### Getting the PDF
Download the full book from:
```
https://openstax.org/details/books/organic-chemistry
```
Then extract chapter 21 (pages 741 to 792) using PyMuPDF:

```python
import fitz
doc = fitz.open("openstax_organic_chemistry.pdf")
sub = fitz.open()
sub.insert_pdf(doc, from_page=740, to_page=791)
sub.save("data/uploads/openstax_ch21_carboxylic_acid_derivatives.pdf")
```

Note: PyMuPDF uses zero-based page indexing, so page 741 corresponds to index 740.

We process only the first 5 pages to limit runtime during development.
Results show detected equations with their LaTeX source and detection strategy.

In [25]:
# Load the ingested document from notebook 1
pdf_path = UPLOADS_DIR / "openstax_ch21_carboxylic_acid_derivatives.pdf"

doc = IngestedDocument(
    input_type  = InputType.PDF,
    text        = "",
    source_path = pdf_path,
)

doc_with_equations = extract_equations_from_pdf(doc, max_pages=20)

print(f"Pages processed : 20")
print(f"Equations found : {len(doc_with_equations.equations)}")
print()

for i, eq in enumerate(doc_with_equations.equations):
    print(f"--- Equation {i+1} ---")
    print(f"Page     : {eq['page'] + 1}")
    print(f"Strategy : {eq['strategy']}")
    print(f"LaTeX    : $$\n{eq['latex']}\n$$")
    print()

Pages processed : 5
Equations found : 1

--- Equation 1 ---
Page     : 19
Strategy : spacing
LaTeX    : $$
\begin{array}{l}{{{}^{\mathrm{culjlpil}}\,1}}\\ {{{}C{\Delta\mathrm{Arb}{\bf J}{\bf J}{\bf J}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{l}}{\bf\hat{\hat{l}}}{\hat{l}}{\bf\hat{\hat{\bf\hat{\hat{l}}}}}{\bf\hat{\hat{\hat{\hat{\hat{\bf\hat{l}}}}}}}}{\hat{\hat{\hat{\hat{\hat{\bf\hat{\hat{\hat{\Lambda}}}}}}}}}{\hat{\bf{{\bf\hat{{\bf{\bf\hat{\hat{\hat{\hat{\bf\hat{\hat{\hat{\hat{\hat{\hat{\hat{\hat{\
$$



### Limitations
1. Font-based detection only works on text-native PDFs; scanned PDFs require OCR preprocessing.
2. pix2tex accuracy degrades significantly on handwritten equations and non-standard notation.
3. Complex multi-line equations are sometimes split into multiple detections.
4. Detection recall is not guaranteed; some equations may be missed by all three strategies.
5. Handwritten equation recognition is best-effort; results should be reviewed manually before use.
6. Automatic equation detection performs poorly on PDFs with complex layouts such as textbooks
   with molecular structure diagrams and multi-column figures. Manual region selection is
   recommended as default; automatic detection is provided as experimental functionality.